# Visualize SCALP-lite Embedding

Set `SCALP_INPUT_H5AD` to your input file. Optionally set `SCALP_OUTPUT_H5AD` for the annotated output.

In [1]:
import os
from pathlib import Path

import pandas as pd

from scalp_lite import read_h5ad, save_h5ad, validate_adata, ensure_pca, build_scalp_graph, embed_graph, plot_embedding_pair

def resolve_path(env_name, default):
    path = Path(os.environ.get(env_name, default))
    if path.exists() or path.is_absolute():
        return path
    notebook_relative = Path("..") / path
    return notebook_relative if notebook_relative.exists() else path

input_path = resolve_path("SCALP_INPUT_H5AD", "data/scanpy-pbmc3k.h5ad")
output_path = Path(os.environ["SCALP_OUTPUT_H5AD"]) if "SCALP_OUTPUT_H5AD" in os.environ else input_path.parent / "scalp_lite_embedded.h5ad"
batch_key = os.environ.get("SCALP_BATCH_KEY", "batch")
label_key = os.environ.get("SCALP_LABEL_KEY", "label")


In [2]:
adata = read_h5ad(input_path)

if batch_key not in adata.obs:
    # PBMC3k is single-batch, so split cells deterministically for a smoke test.
    adata.obs[batch_key] = pd.Categorical([f"split_{i % 3}" for i in range(adata.n_obs)])

if label_key not in adata.obs:
    for candidate in ("louvain", "leiden", "cell_type", "celltype"):
        if candidate in adata.obs:
            adata.obs[label_key] = adata.obs[candidate].astype("category")
            break

validate_adata(adata, batch_key=batch_key, label_key=label_key if label_key in adata.obs else None)
ensure_pca(adata, rep_key="X_pca", n_components=40)
adata


AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'batch', 'label'
    var: 'gene_ids', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [3]:
graph = build_scalp_graph(
    adata,
    rep_key="X_pca",
    batch_key=batch_key,
    n_neighbors=15,
    intra_fraction=0.5,
    n_inter_edges=1,
    assignment_quantile=0.95,
)
adata.obsm["X_scalp"] = embed_graph(graph, method="auto", n_components=2)

/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
plot_embedding_pair(adata, embedding_key="X_scalp", batch_key=batch_key, label_key=label_key);


In [ ]:
# The paired plot above shows the same embedding colored by batch and by label.


In [6]:
save_h5ad(adata, output_path)
output_path

PosixPath('../data/scalp_lite_embedded.h5ad')